# M1 Lab 2 — First Eval (LLM-as-a-Judge), v2

Same harness as v1 (Ascend IQ **Version A** vs **Version B**, judged against the golden-set
criteria in `eval-harness-proof.md`), but this version loops over the full 20-row
`starter-dataset.csv` instead of one hardcoded question, and aggregates a win-rate and
G1–G4 pass-rate per version.

**Setup (Colab secrets):** click the key icon in the left sidebar and add three secrets,
enabling notebook access for each:
```
ANTHROPIC_API_KEY
GOOGLE_API_KEY
LANGSMITH_API_KEY
```
**Data:** run the upload cell below and choose `starter-dataset.csv` when prompted (or place it
at `/content/starter-dataset.csv` beforehand to skip the prompt).

In [ ]:
!pip install -q anthropic google-genai langsmith pandas

In [ ]:
import os
from google.colab import userdata

ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
LANGSMITH_API_KEY = userdata.get("LANGSMITH_API_KEY")

# LangSmith tracing — every traced call below shows up in the LangSmith project below.
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGSMITH_PROJECT"] = "ascend-iq-m1-poc"

from langsmith import traceable
import anthropic
from google import genai

anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
genai_client = genai.Client(api_key=GOOGLE_API_KEY)

GENERATOR_MODEL = "claude-sonnet-5"  # different family than the judge, avoids self-preference bias
JUDGE_MODEL = "gemini-3.6-flash"

## Load the dataset

Uploads `starter-dataset.csv` if it isn't already at `/content/starter-dataset.csv`. Columns:
`id`, `category`, `question`, `source_1`, `source_2`, `source_3`, `correct_answer`.

In [ ]:
import pandas as pd
from google.colab import files

CSV_PATH = "/content/starter-dataset.csv"

if not os.path.exists(CSV_PATH):
    uploaded = files.upload()
    CSV_PATH = next(iter(uploaded))

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows")
df.head()

## System prompts (locked in from `eval-harness-proof.md`)

In [ ]:
SYSTEM_A = """You are Ascend IQ, a market intelligence assistant for VP-level strategists and product
leaders at Fortune 500 companies. You answer questions about competitors' pricing,
positioning, and reviews using only the retrieved source documents provided to you.

Rules:
- Lead with the direct answer or verdict in the first line.
- Follow with a short bulleted list of the supporting facts, each with an inline citation
  (source name) immediately after the claim it supports.
- No preamble, no restated question, no repeated caveats.
- If the retrieved sources don't contain enough information to answer confidently, say so
  explicitly and state what's missing — never infer or estimate a value that isn't sourced."""

SYSTEM_B = """You are Ascend IQ, a market intelligence assistant for VP-level strategists and product
leaders at Fortune 500 companies. You answer questions about competitors' pricing,
positioning, and reviews using only the retrieved source documents provided to you.

Rules:
- State the direct answer or verdict in the first sentence.
- Explain the reasoning in short prose paragraphs, organized by the dimension
  being compared (e.g., fee structure, volume pricing), covering every entity
  asked about on each dimension.
- Cite the source name in parentheses immediately after each factual claim.
- Close with one takeaway sentence that follows only from facts cited above.
  Do not introduce new claims.
- If any value the question requires is absent from the retrieved sources, say
  so and name what is missing. Never infer or estimate an unsourced value."""

GOLDEN_CRITERIA = """Good (PASS requires G1-G4):
- G1. Grounded: every factual claim (figure, name, date, price, quote, attribution) appears
  in the retrieved context, and every conclusion follows from cited facts.
- G2. Answer-first: the first sentence directly answers the question asked, or states that
  the retrieved data can't support an answer.
- G3. Complete: every part of the question is addressed; a comparison covers both entities
  on the same dimensions.
- G4. Gaps named: where the context lacks data the answer needs, the output names what's
  missing and doesn't fill it in.

Never (any one = FAIL):
- N1. A figure, name, or claim not present in the retrieved context.
- N2. An unsupported inference stated as fact.
- N3. Answering about a different entity, segment, or tier than the one asked about.
- N4. A confident answer when the retrieved context is empty or irrelevant.
- N5. Hedging, refusing, or flagging "missing data" that is actually present in the context.
"""

## Row helpers: build the retrieved context + judge prompt for one row

In [ ]:
def build_context(row) -> str:
    parts = []
    for col in ("source_1", "source_2", "source_3"):
        val = row.get(col)
        if isinstance(val, str) and val.strip():
            parts.append(val.strip())
    return "\n\n".join(parts) if parts else "(no sources retrieved)"


def build_judge_prompt(question: str, context: str, output_a: str, output_b: str) -> str:
    return f"""You are grading two candidate answers to the same market-intelligence question
against a strict rubric. Judge each candidate independently against every G and N item.

RUBRIC:
{GOLDEN_CRITERIA}

RETRIEVED CONTEXT:
{context}

QUESTION:
{question}

CANDIDATE A:
{output_a}

CANDIDATE B:
{output_b}

Output exactly these lines, nothing else before or after:
A_G1: PASS or FAIL
A_G2: PASS or FAIL
A_G3: PASS or FAIL
A_G4: PASS or FAIL
A_VIOLATIONS: comma-separated N-codes that apply, or NONE
B_G1: PASS or FAIL
B_G2: PASS or FAIL
B_G3: PASS or FAIL
B_G4: PASS or FAIL
B_VIOLATIONS: comma-separated N-codes that apply, or NONE
WINNER: A or B or TIE"""

## Generate + judge, looped over every row

In [ ]:
import re
import time


@traceable(name="generate_answer")
def generate(system_prompt: str, user_message: str) -> str:
    response = anthropic_client.messages.create(
        model=GENERATOR_MODEL,
        max_tokens=1024,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}],
    )
    # Claude Sonnet 5 can emit a thinking block before the text block, so
    # find the text block instead of assuming content[0].
    for block in response.content:
        if block.type == "text":
            return block.text.strip()
    raise ValueError("No text block found in response.content")


@traceable(name="judge_verdict")
def judge(prompt: str) -> str:
    response = genai_client.models.generate_content(model=JUDGE_MODEL, contents=prompt)
    return response.text.strip()


def parse_field(verdict: str, field: str, default=None):
    match = re.search(rf"{field}:\s*([A-Za-z,]+)", verdict)
    return match.group(1).strip() if match else default


results = []

for _, row in df.iterrows():
    question = row["question"]
    context = build_context(row)
    user_message = f"Retrieved context:\n{context}\n\nQuestion: {question}"

    output_a = generate(SYSTEM_A, user_message)
    output_b = generate(SYSTEM_B, user_message)

    verdict = judge(build_judge_prompt(question, context, output_a, output_b))

    a_gates = [parse_field(verdict, f"A_G{i}") for i in range(1, 5)]
    b_gates = [parse_field(verdict, f"B_G{i}") for i in range(1, 5)]

    results.append({
        "id": row["id"],
        "category": row["category"],
        "a_pass_all": all(g == "PASS" for g in a_gates),
        "b_pass_all": all(g == "PASS" for g in b_gates),
        "a_violations": parse_field(verdict, "A_VIOLATIONS", "NONE"),
        "b_violations": parse_field(verdict, "B_VIOLATIONS", "NONE"),
        "winner": parse_field(verdict, "WINNER", "UNPARSED"),
        "raw_verdict": verdict,
    })

    print(f"row {row['id']:>2} ({row['category']}): winner={results[-1]['winner']}")
    time.sleep(1)  # light rate-limit courtesy

results_df = pd.DataFrame(results)
results_df.head()

## Aggregate: win-rate + G1–G4 pass-rate per version

In [ ]:
n = len(results_df)
win_counts = results_df["winner"].value_counts()

print("Win rate (of", n, "rows):")
print(win_counts.to_string())
print()
print(f"Version A (Concise)   G1-G4 all-pass rate: {results_df['a_pass_all'].mean():.0%}")
print(f"Version B (Narrative) G1-G4 all-pass rate: {results_df['b_pass_all'].mean():.0%}")
print()
print("Win rate by category:")
print(results_df.groupby("category")["winner"].value_counts().to_string())

## Save results

Writes the per-row results (winner, gate pass/fail, raw judge text) to CSV for the
**Screenshots/links** section of `eval-harness-proof.md`, and offers it for download.

In [ ]:
results_df.to_csv("eval-results-v2.csv", index=False)
files.download("eval-results-v2.csv")